# Multiplying two matrices that do not fit on the GPU

Two matrices of about 30 GiB each, multiplied on a 23 GiB card. First the ordinary way,
which cannot work, then with CUDA managed memory, where the driver pages the data between
host RAM and the device as the kernel touches it.

In [1]:
import time

import torch

GiB = 2**30
free, total = torch.cuda.mem_get_info()
print(f"torch {torch.__version__}")
print(f"gpu   {torch.cuda.get_device_properties(0).name}, "
      f"{free / GiB:.2f} / {total / GiB:.2f} GiB free")

# A is N x K, B is K x N, so the product is a small N x N matrix and the two operands
# are the large things. Filled with ones, so every entry of A @ B must come out as K.
N, K = 3000, 2_700_000
each = N * K * 4
print(f"\nA: {N} x {K:,}   {each / GiB:.1f} GiB")
print(f"B: {K:,} x {N}   {each / GiB:.1f} GiB")
print(f"together {2 * each / GiB:.1f} GiB = {2 * each / total:.2f}x the card")

torch 2.13.0+cu132
gpu   NVIDIA RTX PRO 4000 Blackwell, 23.23 / 23.46 GiB free

A: 3000 x 2,700,000   30.2 GiB
B: 2,700,000 x 3000   30.2 GiB
together 60.3 GiB = 2.57x the card


## The ordinary way

In [2]:
torch.cuda.empty_cache()
try:
    A = torch.ones((N, K), dtype=torch.float32, device="cuda")
    B = torch.ones((K, N), dtype=torch.float32, device="cuda")
    C = A @ B
    print(f"worked: C[0,0] = {C[0, 0].item():.4e}")
    del A, B, C
except torch.OutOfMemoryError as exc:
    print(f"OutOfMemoryError: {str(exc).splitlines()[0][:110]}")
torch.cuda.empty_cache()

OutOfMemoryError: CUDA out of memory. Tried to allocate 30.18 GiB. GPU 0 has a total capacity of 23.46 GiB of which 23.23 GiB is


[W828 10:26:33.979756529 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 32400998400 bytes (free: 24939790336, total: 25190662144).
[W828 10:26:33.980391159 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 32400998400 bytes (free: 24939790336, total: 25190662144).


## With managed memory

`torch.cuda._use_uvm()` allocates through `cudaMallocManaged`, so a tensor can be larger
than the card and lives wherever the driver decides moment to moment. The code inside is
unchanged - same `torch.ones`, same `@`.

It is private API in torch 2.13 (hence the underscore) and needs the `cuda-python` package.

In [3]:
torch.cuda.empty_cache()
with torch.cuda._use_uvm():
    t0 = time.perf_counter()
    A = torch.ones((N, K), dtype=torch.float32, device="cuda")
    B = torch.ones((K, N), dtype=torch.float32, device="cuda")
    torch.cuda.synchronize()
    print(f"allocated {2 * each / GiB:.1f} GiB on {A.device} in "
          f"{time.perf_counter() - t0:.2f} s")

    t0 = time.perf_counter()
    C = A @ B
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    print(f"A @ B     {dt:6.2f} s   {2 * N * N * K / dt / 1e12:5.2f} TFLOP/s")
    print(f"C is {tuple(C.shape)}, C[0,0] = {C[0, 0].item():.5e}  "
          f"(expect {float(K):.5e})")
    print(f"all entries correct: {bool((C == float(K)).all().item())}")
    del A, B, C
torch.cuda.empty_cache()

allocated 60.3 GiB on cuda:0 in 5.27 s
A @ B      62.11 s    0.78 TFLOP/s
C is (3000, 3000), C[0,0] = 2.70000e+06  (expect 2.70000e+06)
all entries correct: True


## What it costs

The result is exact - managed memory changes where the bytes live, not the arithmetic.
What it costs is throughput: the operands have to cross PCIe as the kernel walks them, so
the multiply runs at a small fraction of the rate it would reach on resident data. That is
the trade - the computation becomes possible rather than fast.

It holds up as long as each byte is needed roughly once. An algorithm that sweeps its
input repeatedly (an SVD, say) pays the transfer again on every sweep, and managed memory
stops being a workaround and becomes the bottleneck.